In [38]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy import signal
import warnings
import os
import math
import numpy as np
warnings.filterwarnings('ignore')

In [39]:
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 150)

In [40]:
downsample_factor = 1 # to reduce plotly plot memory sizes

In [41]:
while not os.path.isdir(os.path.join(os.getcwd(), 'data')):
    os.chdir("../") # set cwd to root dir

In [42]:
lf_control_df_02 = pd.read_csv("data/DUO-GAIT/processed/OG_st_control/sub_02/left_foot_core_params.csv")
lf_control_df_02.rename({ "timestamps": "start_times" }, axis=1, inplace=True)
lf_control_df_02['Setup'] = 'Control + LF'
lf_control_df_02['Participant'] = 2

rf_control_df_02 = pd.read_csv("data/DUO-GAIT/processed/OG_st_control/sub_02/right_foot_core_params.csv")
rf_control_df_02.rename({ "timestamps": "start_times" }, axis=1, inplace=True)
rf_control_df_02['Setup'] = 'Control + RF'
rf_control_df_02['Participant'] = 2

lf_fatigue_df_02 = pd.read_csv("data/DUO-GAIT/processed/OG_st_fatigue/sub_02/left_foot_core_params.csv")
lf_fatigue_df_02.rename({ "timestamps": "start_times" }, axis=1, inplace=True)
lf_fatigue_df_02['Setup'] = 'Fatigue + LF'
lf_fatigue_df_02['Participant'] = 2

rf_fatigue_df_02 = pd.read_csv("data/DUO-GAIT/processed/OG_st_fatigue/sub_02/right_foot_core_params.csv")
rf_fatigue_df_02.rename({ "timestamps": "start_times" }, axis=1, inplace=True)
rf_fatigue_df_02['Setup'] = 'Fatigue + RF'
rf_fatigue_df_02['Participant'] = 2

rf_fatigue_df_02

,stride_index,start_times,stride_lengths,clearances_min,clearances_max,stride_times,swing_times,stance_times,stance_ratios,fo_times,ic_times,fo_samples,ic_samples,is_outlier,turning_step,turning_interval,interrupted,Setup,Participant
0,0,2.35938,1.508116,0.011088,NaN,1.09375,0.46094,0.63281,0.578569,2.99219,3.45313,383,442,True,False,True,False,Fatigue + RF,2
1,1,3.45313,1.578072,0.014588,0.068251,1.10937,0.47656,0.63281,0.570423,4.08594,4.56250,523,584,False,False,True,False,Fatigue + RF,2
2,2,4.56250,1.723842,0.032961,0.073925,1.12500,0.49218,0.63282,0.562507,5.19532,5.68750,665,728,False,False,False,False,Fatigue + RF,2
3,3,5.68750,1.686841,0.034044,0.078210,1.08594,0.50000,0.58594,0.539569,6.27344,6.77344,803,867,False,False,False,False,Fatigue + RF,2
4,4,6.77344,1.531519,0.015682,0.064903,1.08594,0.49219,0.59375,0.546761,7.36719,7.85938,943,1006,False,False,False,False,Fatigue + RF,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
327,327,370.25000,1.541911,0.014123,0.064523,1.08594,0.49219,0.59375,0.546761,370.84375,371.33594,47468,47531,False,False,False,False,Fatigue + RF,2
328,328,371.33594,1.535646,0.022723,0.069562,1.04688,0.46094,0.58594,0.559701,371.92188,372.38282,47606,47665,False,False,False,False,Fatigue + RF,2
329,329,372.38282,1.511071,0.009347,0.063607,1.07031,0.46094,0.60937,0.569340,372.99219,373.45313,47743,47802,False,False,False,False,Fatigue + RF,2
330,330,373.45313,1.676061,0.017923,0.066817,1.07812,0.47656,0.60156,0.557971,374.05469,374.53125,47879,47940,False,False,True,False,Fatigue + RF,2


In [43]:
def create_start_end_samples_strides(df, pid, is_control):
    df.sort_values(by='stride_index', inplace=True)
    df['start_samples'] = df['ic_samples'].shift(1)
    target_time = df.loc[0, 'start_times']
    
    protocol = "control" if is_control else "fatigue"

    fatigue_df = pd.read_csv(f"data/DUO-GAIT/interim/OG_st_{protocol}/sub_{pid:02}/LF.csv")
    fatigue_df.rename({ "timestamp": "Time (secs)", "Unnamed: 0": "Sample" }, axis=1, inplace=True)
    fatigue_df['Delta (secs)'] = fatigue_df['Time (secs)'] - fatigue_df['Time (secs)'].min() # delta time

    ts_eq_check = fatigue_df['Delta (secs)'].apply(lambda x: math.isclose(x, target_time, rel_tol=1e-5))
    start_sample = fatigue_df[ts_eq_check]['Sample'].item() - fatigue_df['Sample'].min()

    df.loc[0, 'start_samples'] = start_sample
    df['start_samples'] = df['start_samples'] + fatigue_df['Sample'].min()
    df['end_samples'] = df['ic_samples'] + fatigue_df['Sample'].min() - 1 # make it inclusive for ending samples too

    df['start_samples'] = df['start_samples'].astype(np.int64)
    df['end_samples'] = df['end_samples'].astype(np.int64)

In [44]:
create_start_end_samples_strides(lf_control_df_02, 2, 1)
create_start_end_samples_strides(lf_fatigue_df_02, 2, 0)

In [45]:
lf_control_df_02 = lf_control_df_02[lf_control_df_02['is_outlier']==False].reset_index(drop=True)
lf_fatigue_df_02 = lf_fatigue_df_02[lf_fatigue_df_02['is_outlier']==False].reset_index(drop=True)

In [46]:
sensor_location = "RL"

In [47]:
control_df_02 = pd.read_csv(f"data/DUO-GAIT/interim/OG_st_control/sub_02/{sensor_location}.csv")
control_df_02.rename({ "timestamp": "Time (secs)", "Unnamed: 0": "Sample" }, axis=1, inplace=True)
control_df_02['AccM'] = np.linalg.norm(control_df_02[['AccX', 'AccY', 'AccZ']].values, axis=1)
control_df_02['GyrM'] = np.linalg.norm(control_df_02[["GyrX", "GyrY", "GyrZ"]].values, axis=1)

fatigue_df_02 = pd.read_csv(f"data/DUO-GAIT/interim/OG_st_fatigue/sub_02/{sensor_location}.csv")
fatigue_df_02.rename({ "timestamp": "Time (secs)", "Unnamed: 0": "Sample" }, axis=1, inplace=True)
fatigue_df_02['AccM'] = np.linalg.norm(fatigue_df_02[['AccX', 'AccY', 'AccZ']].values, axis=1)
fatigue_df_02['GyrM'] = np.linalg.norm(fatigue_df_02[["GyrX", "GyrY", "GyrZ"]].values, axis=1)

**Filter away samples for control and fatigue walk for a non outlier stride**

In [48]:
stride_index = 10
control_df_02 = control_df_02[(control_df_02['Sample']>=lf_control_df_02.loc[stride_index, 'start_samples']) & (control_df_02['Sample']<=lf_control_df_02.loc[stride_index, 'end_samples'])]
fatigue_df_02 = fatigue_df_02[(fatigue_df_02['Sample']>=lf_fatigue_df_02.loc[stride_index, 'start_samples']) & (fatigue_df_02['Sample']<=lf_fatigue_df_02.loc[stride_index, 'end_samples'])]

In [49]:
fig = px.line(control_df_02.iloc[::downsample_factor], x='Time (secs)', y=["AccX", "AccY", "AccZ"], title=f"Avg Male Acc Stride {stride_index+1} Plot for Control ({sensor_location})")
fig.show()

fig = px.line(fatigue_df_02.iloc[::downsample_factor], x='Time (secs)', y=["AccX", "AccY", "AccZ"], title=f"Avg Male Acc Stride {stride_index+1} Plot for Fatigue ({sensor_location})")
fig.show()

fig = px.line(control_df_02.iloc[::downsample_factor], x='Time (secs)', y=["GyrX", "GyrY", "GyrZ"], title=f"Avg Male Gyr Stride {stride_index+1} Plot for Control ({sensor_location})")
fig.show()

fig = px.line(fatigue_df_02.iloc[::downsample_factor], x='Time (secs)', y=["GyrX", "GyrY", "GyrZ"], title=f"Avg Male Gyr Stride {stride_index+1} Plot for Fatigue ({sensor_location})")
fig.show()

Signals are very faint and nearly nonexistant across the fatigue states

In [50]:
fig = px.line(control_df_02.iloc[::downsample_factor], x='Time (secs)', y=["AccM"], title=f"Avg Male Acc Stride {stride_index+1} Magnitude Plot for Control ({sensor_location})")
fig.show()

fig = px.line(fatigue_df_02.iloc[::downsample_factor], x='Time (secs)', y=["AccM"], title=f"Avg Male Acc Stride {stride_index+1} Magnitude Plot for Fatigue ({sensor_location})")
fig.show()

fig = px.line(control_df_02.iloc[::downsample_factor], x='Time (secs)', y=["GyrM"], title=f"Avg Male Gyr Stride {stride_index+1} Magnitude Plot for Control ({sensor_location})")
fig.show()

fig = px.line(fatigue_df_02.iloc[::downsample_factor], x='Time (secs)', y=["GyrM"], title=f"Avg Male Gyr Stride {stride_index+1} Magnitude Plot for Fatigue ({sensor_location})")
fig.show()

Acc + Gyr magnitude signals for left wrist look damped when fatigued. This could be useful to the model. Acc looks a bit more variable when fatigued and more smoother during control. The signal is small but it exists for left leg. For left foot, acc plots also show more variability. 

In [51]:
def butter_lowpass_filter(series, cutoff=10.0, fs=128.0, order=3):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = signal.butter(order, normal_cutoff, btype='low', analog=False)
    y = signal.filtfilt(b, a, series)
    return y

**Apply butterworth filter from reference for visualization**

In [52]:
for sensor in ["Acc", "Gyr"]:
    for axis in ["X", "Y", "Z", "M"]:
        col = f"{sensor}{axis}"
        filtered_col = f"{col}F"

        control_df_02[filtered_col] = butter_lowpass_filter(control_df_02[col])
        fatigue_df_02[filtered_col] = butter_lowpass_filter(fatigue_df_02[col])

In [53]:
fig = px.line(control_df_02.iloc[::downsample_factor], x='Time (secs)', y=["AccX", "AccXF"], title=f"Butterworth Filter for Control ({sensor_location})")
fig.update_layout(yaxis_title="Acceleration")
fig.show()

This particular butterworth filter flattens all the spikes and smoothes them out. 

**Visualize how signals change after application of Butterworth filter**

In [54]:
fig = px.line(control_df_02.iloc[::downsample_factor], x='Time (secs)', y=["AccXF", "AccYF", "AccZF"], title=f"Avg Male Acc Stride {stride_index+1} Plot for Control ({sensor_location})")
fig.show()

fig = px.line(fatigue_df_02.iloc[::downsample_factor], x='Time (secs)', y=["AccXF", "AccYF", "AccZF"], title=f"Avg Male Acc Stride {stride_index+1} Plot for Fatigue ({sensor_location})")
fig.show()

fig = px.line(control_df_02.iloc[::downsample_factor], x='Time (secs)', y=["GyrXF", "GyrYF", "GyrZF"], title=f"Avg Male Gyr Stride {stride_index+1} Plot for Control ({sensor_location})")
fig.show()

fig = px.line(fatigue_df_02.iloc[::downsample_factor], x='Time (secs)', y=["GyrXF", "GyrYF", "GyrZF"], title=f"Avg Male Gyr Stride {stride_index+1} Plot for Fatigue ({sensor_location})")
fig.show()

In [55]:
fig = px.line(control_df_02.iloc[::downsample_factor], x='Time (secs)', y=["AccMF"], title=f"Avg Male Acc Stride {stride_index+1} Magnitude Plot for Control ({sensor_location})")
fig.show()

fig = px.line(fatigue_df_02.iloc[::downsample_factor], x='Time (secs)', y=["AccMF"], title=f"Avg Male Acc Stride {stride_index+1} Magnitude Plot for Fatigue ({sensor_location})")
fig.show()

fig = px.line(control_df_02.iloc[::downsample_factor], x='Time (secs)', y=["GyrMF"], title=f"Avg Male Gyr Stride {stride_index+1} Magnitude Plot for Control ({sensor_location})")
fig.show()

fig = px.line(fatigue_df_02.iloc[::downsample_factor], x='Time (secs)', y=["GyrMF"], title=f"Avg Male Gyr Stride {stride_index+1} Magnitude Plot for Fatigue ({sensor_location})")
fig.show()

In [56]:
def random_scaling(input, sigma=1.2):
    input = input[np.newaxis, :]
    scalingFactor = np.random.normal(loc=1.0, scale=sigma, size=(input.shape[0], 1)).astype(input.dtype, copy=False)
    myNoise = np.matmul(scalingFactor, np.ones((1, input.shape[1]), dtype=input.dtype))
    return input * myNoise

In [57]:
for sensor in ["Acc", "Gyr"]:
    for axis in ["X", "Y", "Z", "M"]:
        col = f"{sensor}{axis}"
        filtered_col = f"{col}RS"
        
        control_df_02[filtered_col] = random_scaling(control_df_02[col].to_numpy()).flatten()
        fatigue_df_02[filtered_col] = random_scaling(fatigue_df_02[col].to_numpy()).flatten()

In [58]:
fig = px.line(control_df_02.iloc[::downsample_factor], x='Time (secs)', y=["AccX", "AccXRS"], title=f"Random Scaling Augmentation for Control ({sensor_location})")
fig.update_layout(yaxis_title="Acceleration")
fig.show()